# Lab 02 — Text Preprocessing for LLMs (NCA-GENL delta)

⚠️ **First run needs internet**: the two tokenizer cells download `gpt2` (BPE) and `bert-base-uncased` (WordPiece) from the Hugging Face Hub (small, then cached). Everything else is offline.

Structure verified; the download cells could not be executed in the authoring environment — if a `from_pretrained` cell fails, check connectivity/proxy first.

In [ ]:
# Environment check — runs as-is
import transformers, datasets
from transformers import AutoTokenizer
print("transformers", transformers.__version__, "| datasets", datasets.__version__)

In [ ]:
# Downloads on first run, cached afterwards
tok_bpe = AutoTokenizer.from_pretrained("gpt2")               # byte-level BPE
tok_wp  = AutoTokenizer.from_pretrained("bert-base-uncased")  # WordPiece
print("vocab sizes — gpt2:", tok_bpe.vocab_size, "| bert:", tok_wp.vocab_size)

## 1. Same sentence, two tokenizers

In [ ]:
# Runs as-is: tricky strings — rare word, numbers, code, non-English
samples = [
    "The indefatigable ossobuco cost 1,249.99 euros in Milano.",
    "for i in range(10): x += i**2  # accumulate",
    "Straße, naïveté, 東京, ½ cup flour",
]
for s in samples:
    print("-" * 60, "\n", s)
    print("  BPE      :", tok_bpe.tokenize(s))
    print("  WordPiece:", tok_wp.tokenize(s))
    print("  counts   : BPE", len(tok_bpe.tokenize(s)), "| WP", len(tok_wp.tokenize(s)))

In [ ]:
# TODO: pick 3 sentences of your own (include your domain's jargon). Compare splits
# and token counts. Then write one paragraph: how vocab choice shifts effective
# context length (more tokens per sentence = fewer sentences per context window)
# and multilingual behavior (byte-level BPE never OOVs; WordPiece falls back to
# [UNK] / heavy fragmentation on unseen scripts).


## 2. Batching: padding, truncation, attention masks

In [ ]:
# Runs as-is: what the mask actually masks
batch = ["Short one.", "A much, much longer sentence that will dominate the batch length entirely."]
enc = tok_wp(batch, padding=True, truncation=True, return_tensors="pt")
print("input_ids shape:", enc["input_ids"].shape)
print(enc["attention_mask"])

In [ ]:
# TODO: re-encode with padding="max_length", max_length=64 and compare tensor
# shapes vs dynamic (longest-in-batch) padding. State the compute tradeoff:
# fixed-length wastes FLOPs on pad tokens but gives static shapes (nice for
# compilers/serving); dynamic padding saves compute but shapes vary per batch.
# Bonus: sort-by-length bucketing as the standard middle ground.


## 3. When classical text cleaning is wrong for LLMs

In [ ]:
# TODO (discriminator the exam likes): take one sample sentence, apply classic
# cleaning (lowercase, strip punctuation, remove stopwords — plain Python is fine),
# then tokenize the cleaned vs raw version with tok_bpe.
# Answer: why does classic cleaning help bag-of-words/TF-IDF pipelines but HURT
# pretrained-LLM pipelines? (Tokenizers were trained on raw-ish text; casing and
# punctuation carry signal; cleaning creates train/inference mismatch.)


### Exit criteria touched here
- [ ] BPE vs WordPiece differences on a concrete sentence I ran
- [ ] padding + attention masks, and the fixed-vs-dynamic padding tradeoff
- [ ] when classical text cleaning helps vs hurts an LLM pipeline